# **Exploración**

## **Librerías**

In [1]:
import pandas as pd
import numpy as np

In [74]:
import os
import re
import fitz
import unicodedata

In [8]:
from tqdm import tqdm

## **Variables Globales**

In [19]:
pdf_file = '../Data/Raw/El_Ojo_Del_Mundo.pdf'
text_file = '../Data/Processed/El_Ojo_Del_Mundo.txt'

## **Limpieza de la data**

### **Transformación del PDF**

In [ ]:
doc = fitz.open(pdf_file)

# Convertimos el archivo en .txt
with open(text_file, 'w', encoding='utf-8') as file:
    for page in tqdm(doc, desc='Extrayendo texto', unit='página'):
        text = page.get_text()
        file.write(text)

Extrayendo texto: 100%|██████████| 762/762 [00:02<00:00, 323.71página/s]


In [128]:
# Cargamos nuestro .txt
with open(text_file, 'r', encoding='utf-8') as file:
    text = file.read()

In [129]:
print(text[:500])

www.lectulandia.com - Página 1
 
El mundo gira y gira y una vez tras otra vuelven, al cabo de los años, de los
milenios, los días prometidos en las leyendas. Desde el horrible momento en
el que Lews Therin, que fue Señor de la Luz, Impulsado por las fuerzas de la
Oscuridad, dio muerte a todos los suyos, no ha cesado la lucha entre la luz y
las tinieblas, la vida y la muerte, el bien y el mal.
En una pequeña aldea solitaria de la región de Dos Rios vive Rand, un joven
granjero, en compañía de su 


In [130]:
# Eliminamos los pies de página
clean_text = re.sub(r"www\.lectulandia\.com\s*-\s*Página\s*\d+", '', text)

In [131]:
# Quitamos las tildes
clean_text = unicodedata.normalize('NFKD', clean_text).encode('ascii', 'ignore').decode('utf-8')

In [132]:
print(clean_text[:500])


 
El mundo gira y gira y una vez tras otra vuelven, al cabo de los anos, de los
milenios, los dias prometidos en las leyendas. Desde el horrible momento en
el que Lews Therin, que fue Senor de la Luz, Impulsado por las fuerzas de la
Oscuridad, dio muerte a todos los suyos, no ha cesado la lucha entre la luz y
las tinieblas, la vida y la muerte, el bien y el mal.
En una pequena aldea solitaria de la region de Dos Rios vive Rand, un joven
granjero, en compania de su padre. Una noche son asaltados


### **Separación del texto por capitulos**

In [133]:
# Búsqueda de los capítulos
patron_capitulos = r"\n\s*(\d{1,2})\s+([^\n]+)"
coincidencias = list(re.finditer(patron_capitulos, clean_text))

In [134]:
# Búsqueda del glosario
inicio_glosario = re.search('glosario', clean_text)

In [136]:
# Sabemos que el primer libro tiene 53 capítulos.
total_esperado = 53

if len(coincidencias) == total_esperado:
    print(f'El número de capítulos identificados coincide con el esperado: {len(coincidencias)}')
else:
    print(f'El número de capítulos identificados no coincide con el esperado: {len(coincidencias)}')

El número de capítulos identificados coincide con el esperado: 53


In [138]:
for i in range(len(coincidencias)):
    
    ini_cap = coincidencias[i].start()
    fin_cap = coincidencias[i + 1].start() if i + 1 < len(coincidencias) else len(clean_text)
    
    if fin_cap > inicio_glosario.start():
        fin_cap = inicio_glosario.start()

    titulo = coincidencias[i].group(2).replace(' ', '_').lower()
    
    num_archivo = i + 1 if len(str(i + 1)) == 2 else f'0{i+1}'
    nombre_archivo = f'{num_archivo}.{titulo}.txt'
    
    with open(f'../Data/Clean/01_EOdM/{nombre_archivo}', 'w', encoding='utf-8') as file:
        file.write(clean_text[ini_cap:fin_cap])